In [1]:
import scanpy as sc
import numpy as np
import anndata as ad
from scipy.sparse import csr_matrix
import glob
import pandas as pd
import os


def get_anndata_her2st(sample):
    img_path = os.path.join(sample, [f for f in os.listdir(sample) if f.endswith('.jpg') and f != 'spot_view.jpg'][0])
    
    spots_coord = pd.read_csv(os.path.join(sample, "spots.csv"), index_col=0)
    cnt = pd.read_csv(os.path.join(sample, "stdata.csv"), index_col=0)
    
    spots_coord = spots_coord.loc[list(set(cnt.index)&set(spots_coord.index))]
    cnt = cnt.loc[list(set(cnt.index)&set(spots_coord.index))]

    location = np.matrix(spots_coord.values)
    Xdense = np.matrix(cnt.values)
    
    batch = np.array([sample.split('/')[-1] for i in range(Xdense.shape[0])])

    X = csr_matrix(Xdense, dtype=np.float32)
    adata_vis = ad.AnnData(X, obsm={"spatial": location}, obs={"sample": batch})
    adata_vis.obs_names = [sample.split('/')[-1] + '_' + idx for idx in list(spots_coord.index)]
    # adata_vis.obs_names = list(spots_coord.index)
    adata_vis.var_names = list(cnt.columns)
    return adata_vis

samples_path = [path for path in glob.glob("../../data/stnet/*") if not path.endswith(".h5ad") and not path.endswith(".ipynb")]

adata_list = []
from tqdm import tqdm
for i in tqdm(range(0, len(samples_path))):
    adata_1 = get_anndata_her2st(samples_path[i])
    adata_list.append(adata_1)

adata_vis = ad.concat(adata_list)

  4%|▍         | 3/68 [00:21<07:37,  7.04s/it]


ParserError: Error tokenizing data. C error: Calling read(nbytes) on source failed. Try engine='python'.

In [2]:
import pandas as pd

results_folder = '../../data/breast_sc/cell2location_res'
ref_run_name = f'{results_folder}/breast_reference_signatures'
inf_aver = pd.read_csv(f"{ref_run_name}/inf_aver.csv", index_col=0)
inf_aver

,fibroblast,T cell,mast cell,macrophage,vascular associated smooth muscle cell,CD4-positive helper T cell,lymphocyte,natural killer cell,"CD4-positive, alpha-beta T cell",basal cell,...,class switched memory B cell,IgG plasma cell,IgA plasma cell,conventional dendritic cell,endothelial cell of lymphatic vessel,capillary endothelial cell,luminal epithelial cell of mammary gland,mammary gland epithelial cell,vein endothelial cell,endothelial cell of artery
ENSG00000187634,0.016341,0.003520,0.000729,0.000525,0.000805,0.000034,0.001233,0.000295,0.000218,0.000394,...,0.000222,0.006724,0.004025,0.000890,0.000852,0.001395,0.004111,0.000428,0.000224,0.000118
ENSG00000188976,0.117370,0.146280,0.016717,0.073333,0.051991,0.039917,0.051192,0.049764,0.043392,0.171738,...,0.042114,0.026784,0.040397,0.065859,0.053536,0.109784,0.256512,0.276506,0.165131,0.051012
ENSG00000187583,0.000311,0.005345,0.000754,0.003718,0.000774,0.003845,0.008622,0.000531,0.001245,0.003728,...,0.000648,0.021841,0.023673,0.004289,0.000818,0.000226,0.051024,0.063262,0.000308,0.000094
ENSG00000188290,0.102318,0.060894,0.029283,0.107470,1.194010,0.007908,0.039131,0.012359,0.001107,0.012373,...,0.003083,0.004363,0.005516,0.153251,0.294717,0.089528,0.950762,0.654805,0.069212,0.339997
ENSG00000187608,0.148548,0.641984,0.027741,2.256322,0.478522,0.162423,0.235466,0.186748,0.154152,0.004705,...,0.052665,0.103493,0.156098,0.438484,0.182963,0.565770,0.097423,0.059964,0.560094,0.859192
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSG00000198695,0.150368,0.914952,0.047736,0.389017,0.104164,0.220048,0.218192,0.131441,0.310187,0.237162,...,0.240536,0.282472,0.556094,0.408530,0.208988,0.410932,1.071885,0.934500,0.439132,0.181709
ENSG00000198727,17.179066,18.372324,9.565456,22.042368,14.920470,6.843974,8.016482,6.811477,11.395451,16.847425,...,10.901145,12.678282,19.543940,21.561310,11.723537,16.444237,33.896400,45.283203,23.438406,9.522370
ENSG00000277836,0.005831,0.948746,1.011197,1.117903,0.377961,0.614520,1.003062,0.849213,0.799271,0.100614,...,1.026887,0.870512,0.867732,1.027124,0.826734,0.537902,0.126450,0.566995,0.468169,0.868281
ENSG00000277856,0.000010,0.913959,0.722972,0.660645,0.000369,0.000196,0.688798,0.242963,0.126896,0.000023,...,0.354873,0.953489,0.528784,0.422856,0.007974,0.000129,0.000016,0.000074,0.000056,0.001172


In [3]:
adata_ref = sc.read_h5ad("../../data/breast_sc/cell2location_res/breast_reference_signatures/breast_sc.h5ad")
gene_name_df = pd.DataFrame(data=adata_ref.var['feature_name'].index, index=adata_ref.var['feature_name'])
gene_name_df

,0
feature_name,
SAMD11,ENSG00000187634
NOC2L,ENSG00000188976
PLEKHN1,ENSG00000187583
HES4,ENSG00000188290
ISG15,ENSG00000187608
...,...
MT-ND6,ENSG00000198695
MT-CYB,ENSG00000198727
ENSG00000277836.1,ENSG00000277836


In [4]:
# 计算所有基因在所有细胞中的平均表达量，并确保结果是一维数组
if isinstance(adata_vis.X, np.ndarray):
    mean_expression = adata_vis.X.mean(axis=0)
else:
    # 对于稀疏矩阵，先转换为一维数组
    mean_expression = np.ravel(adata_vis.X.mean(axis=0))

# 确保mean_expression是一维数组
mean_expression = np.squeeze(np.asarray(mean_expression))

# 将平均表达量赋值给var
adata_vis.var['mean_expression'] = mean_expression

# 对基因按照平均表达量进行排序
adata_vis.var.sort_values(by='mean_expression', ascending=False, inplace=True)

# 保留表达量最高的250个基因
top_genes = adata_vis.var.head(2000).index

# 使用这250个基因过滤原始的 AnnData 对象
adata_vis = adata_vis[:, top_genes]

# adata_filtered 现在包含了表达量最高的250个基因的信息
adata_vis

View of AnnData object with n_obs × n_vars = 30655 × 2000
    obs: 'sample'
    var: 'mean_expression'
    obsm: 'spatial'

In [5]:
# find shared genes and subset both anndata and reference signatures
intersect = np.intersect1d(adata_vis.var_names, adata_ref.var['feature_name'])

adata_vis = adata_vis[:, intersect].copy()
gene_id = gene_name_df.loc[adata_vis.var_names]
adata_vis.var_names = gene_id[0]
inf_aver = inf_aver.loc[adata_vis.var_names, :].copy()

intersect

array([], dtype=object)

In [6]:
gpu_list = [0]
gpu_list_str = ','.join(map(str, gpu_list))
os.environ.setdefault("CUDA_VISIBLE_DEVICES", gpu_list_str)

'0'

In [7]:
import cell2location

# prepare anndata for cell2location model
cell2location.models.Cell2location.setup_anndata(adata=adata_vis, batch_key="sample")
# create and train the model
mod = cell2location.models.Cell2location(
    adata_vis, cell_state_df=inf_aver, 
    # the expected average cell abundance: tissue-dependent 
    # hyper-prior which can be estimated from paired histology:
    N_cells_per_location=30,
    # hyperparameter controlling normalisation of
    # within-experiment variation in RNA detection:
    detection_alpha=20
) 
mod.view_anndata_setup()

/home/wqzhao/anaconda3/envs/gene/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/wqzhao/anaconda3/envs/gene/lib/python3.11/site-packages/cell2location/models/_cell2location_model.py:91: RuntimeWarning: invalid value encountered in scalar divide
  self.detection_mean_ = (sp_total / model_kwargs.get("N_cells_per_location", 1)) / sc_total


Anndata setup with scvi-tools version 1.1.2.

Setup via `Cell2location.setup_anndata` with arguments:

{
│   'layer': None,
│   'batch_key': 'sample',
│   'labels_key': None,
│   'categorical_covariate_keys': None,
│   'continuous_covariate_keys': None
}

         Summary Statistics         
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃     Summary Stat Key     ┃ Value ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│         n_batch          │  68   │
│         n_cells          │ 30655 │
│ n_extra_categorical_covs │   0   │
│ n_extra_continuous_covs  │   0   │
│         n_labels         │   1   │
│          n_vars          │   0   │
└──────────────────────────┴───────┘

               Data Registry                
┏━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Registry Key ┃    scvi-tools Location    ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│      X       │          adata.X          │
│    batch     │ adata.obs['_scvi_batch']  │
│    ind_x     │   adata.obs['_indices']   │
│    labels    │ adata.obs['_scvi_labels'] │
└──────────────┴───────────────────────────┘

                   batch State Registry                   
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃   Source Location   ┃ Categories ┃ scvi-tools Encoding ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ adata.obs['sample'] │  23209_C1  │          0          │
│                     │  23209_C2  │          1          │
│                     │  23209_D1  │          2          │
│                     │  23268_C1  │          3          │
│                     │  23268_C2  │          4          │
│                     │  23268_D1  │          5          │
│                     │  23269_C1  │          6          │
│                     │  23269_C2  │          7          │
│                     │  23269_D1  │          8          │
│                     │  23270_D2  │          9          │
│                     │  23270_E1  │         10          │
│                     │  23270_E2  │         11          │
│                     │  23272_D2  │         12          │
│                     │  23272_E1  │         13          │
│                     │  23272_E2  │         14          │
│                     │  23277_D2  │         15          │
│                     │  23277_E1  │         16          │
│                     │  23277_E2  │         17          │
│                     │  23287_C1  │         18          │
│                     │  23287_C2  │         19          │
│                     │  23287_D1  │         20          │
│                     │  23288_D2  │         21          │
│                     │  23288_E1  │         22          │
│                     │  23288_E2  │         23          │
│                     │  23377_C1  │         24          │
│                     │  23377_C2  │         25          │
│                     │  23377_D1  │         26          │
│                     │  23450_D2  │         27          │
│                     │  23450_E1  │         28          │
│                     │  23450_E2  │         29          │
│                     │  23506_C1  │         30          │
│                     │  23506_C2  │         31          │
│                     │  23506_D1  │         32          │
│                     │  23508_D2  │         33          │
│                     │  23508_E1  │         34          │
│                     │  23508_E2  │         35          │
│                     │  23567_D2  │         36          │
│                     │  23567_E1  │         37          │
│                     │  23567_E2  │         38          │
│                     │  23803_D2  │         39          │
│                     │  23803_E1  │         40          │
│                     │  23803_E2  │         41          │
│                     │  23810_D2  │         42          │
│                     │  23810_E1  │         43          │
│                     │  23810_E2  │         44          │
│                     │  23895_C1  │         45          │
│                     │  23895_C2  │         46          │
│                     │  23895_D1  │         47          │
│                     │  23901_C2  │         48          │
│                     │  23901_D1  │         49          │
│                     │  23903_C1  │         50          │
│                     │  23903_C2  │         51          │
│                     │  23903_D1  │         52          │
│                     │  23944_D2  │         53          │
│                     │  23944_E1  │         54          │
│                     │  23944_E2  │         55          │
│                     │  24044_D2  │         56          │
│                     │  24044_E1  │         57          │
│                     │  24044_E2  │         58          │
│                     │  24105_C1  │         59          │
│                     │  24105_C2  │         60          │
│                     │  24105_D1  │         61          │
│                     │  24220_D2  │         62          │
│                     │  24220_E1  │         63

                     labels State Registry                      
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃      Source Location      ┃ Categories ┃ scvi-tools Encoding ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ adata.obs['_scvi_labels'] │     0      │          0          │
└───────────────────────────┴────────────┴─────────────────────┘

In [9]:
from matplotlib import pyplot as plt


mod.train(max_epochs=30000,
          # train using full data (batch_size=None)
          batch_size=None,
          # use all data points in training because
          # we need to estimate cell abundance at all locations
          train_size=1,
        #   use_gpu=True,
         )

# plot ELBO loss history during training, removing first 100 epochs from the plot
mod.plot_history(1000)
plt.legend(labels=['full data training'])

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


/home/wqzhao/anaconda3/envs/gene/lib/python3.11/site-packages/lightning/pytorch/trainer/configuration_validator.py:72: You passed in a `val_dataloader` but have no `validation_step`. Skipping val loop.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/wqzhao/anaconda3/envs/gene/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:441: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=79` in the `DataLoader` to improve performance.
/home/wqzhao/anaconda3/envs/gene/lib/python3.11/site-packages/lightning/pytorch/loops/fit_loop.py:293: The number of training batches (1) is smaller than the logging interval Trainer(log_every_n_steps=10). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


ValueError: Expected parameter rate (Tensor of shape (1, 1)) of distribution Gamma(concentration: tensor([[10.]], device='cuda:0'), rate: tensor([[nan]], device='cuda:0')) to satisfy the constraint GreaterThan(lower_bound=0.0), but found invalid values:
tensor([[nan]], device='cuda:0')
               Trace Shapes:                 
                Param Sites:                 
               Sample Sites:                 
               m_g_mean dist          |  1  1
                       value          |  1  1
        m_g_alpha_e_inv dist          |  1  1
                       value          |  1  1
                    m_g dist          |  1  0
                       value          |  1  0
 n_s_cells_per_location dist 30655  1 |      
                       value 30655  1 |      
b_s_groups_per_location dist 30655  1 |      
                       value 30655  1 |      
    z_sr_groups_factors dist 30655 50 |      
                       value 30655 50 |      
 k_r_factors_per_groups

In [ ]:
# In this section, we export the estimated cell abundance (summary of the posterior distribution).
adata_vis = mod.export_posterior(
    adata_vis, sample_kwargs={
        'num_samples': 1000, 
        'batch_size': mod.adata.n_obs, 
        # 'use_gpu': True
    }
)

Sampling global variables, sample: 100%|██████████| 999/999 [00:16<00:00, 62.14it/s]


In [ ]:
adata_vis.obsm['q05_cell_abundance_w_sf']

,q05cell_abundance_w_sf_fibroblast,q05cell_abundance_w_sf_T cell,q05cell_abundance_w_sf_mast cell,q05cell_abundance_w_sf_macrophage,q05cell_abundance_w_sf_vascular associated smooth muscle cell,q05cell_abundance_w_sf_CD4-positive helper T cell,q05cell_abundance_w_sf_lymphocyte,q05cell_abundance_w_sf_natural killer cell,"q05cell_abundance_w_sf_CD4-positive, alpha-beta T cell",q05cell_abundance_w_sf_basal cell,...,q05cell_abundance_w_sf_class switched memory B cell,q05cell_abundance_w_sf_IgG plasma cell,q05cell_abundance_w_sf_IgA plasma cell,q05cell_abundance_w_sf_conventional dendritic cell,q05cell_abundance_w_sf_endothelial cell of lymphatic vessel,q05cell_abundance_w_sf_capillary endothelial cell,q05cell_abundance_w_sf_luminal epithelial cell of mammary gland,q05cell_abundance_w_sf_mammary gland epithelial cell,q05cell_abundance_w_sf_vein endothelial cell,q05cell_abundance_w_sf_endothelial cell of artery
B4_10x14,0.322745,0.127396,0.054359,0.144094,0.064218,0.067625,0.120158,0.067951,0.082630,0.113223,...,0.067704,0.057280,0.067174,0.053769,0.112214,0.108073,0.662379,0.221439,0.100400,0.076281
B4_10x15,0.086793,0.062108,0.032637,0.045076,0.035731,0.036261,0.036199,0.037269,0.032493,0.049691,...,0.036758,0.035074,0.020313,0.028462,0.056850,0.078507,0.496786,0.150369,0.067075,0.043068
B4_10x16,0.352340,0.080801,0.045437,0.069823,0.050067,0.047922,0.053163,0.057809,0.049236,0.077856,...,0.055678,0.138083,0.044408,0.049230,0.118602,0.067460,0.809001,0.310326,0.058782,0.059251
B4_10x17,0.142472,0.082922,0.035803,0.073673,0.046139,0.046082,0.046872,0.049118,0.042620,0.083864,...,0.048243,0.026441,0.019896,0.032584,0.063964,0.064812,0.546769,0.249611,0.056769,0.053981
B4_10x18,0.592857,0.397323,0.200663,0.324936,0.318229,0.199330,0.217417,0.218043,0.186787,0.326430,...,0.202283,0.262917,0.188221,0.180672,0.347340,0.453895,2.346975,1.182010,0.411139,0.322390
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
A2_9x26,1.353406,0.889480,0.224394,0.626101,0.330428,0.201800,0.239552,0.252115,0.227664,0.386283,...,0.257492,0.147575,0.194701,0.212763,0.665958,0.347405,2.939570,1.127423,0.355323,0.316699
A2_9x27,0.224888,0.093284,0.044564,0.145704,0.052592,0.052768,0.061047,0.075240,0.050639,0.068681,...,0.048686,0.026877,0.024029,0.056414,0.095237,0.097674,0.881100,0.297026,0.108508,0.079556
A2_9x28,1.132862,0.608820,0.256100,0.609266,0.276234,0.291313,0.365650,0.340795,0.351387,0.403183,...,0.346367,0.411467,0.227544,0.367291,0.704993,0.557279,3.289738,1.353388,0.487213,0.394654
A2_9x29,0.851170,0.497804,0.186640,0.457691,0.245798,0.220004,0.236632,0.229479,0.209693,0.289994,...,0.216962,0.212007,0.116995,0.205807,0.596444,0.446305,2.401990,1.338687,0.475183,0.321421


In [ ]:
run_name = f'{results_folder}/stnet'

# Save model
mod.save(f"{run_name}", overwrite=True)
# Save anndata object with results
adata_file = f"{run_name}/sp_from_top2000.h5ad"
adata_vis.write(adata_file)
adata_file

'../../data/breast_sc/cell2location_res/her2st/sp_from_top2000.h5ad'

In [ ]:
import scanpy as sc

adata = sc.read(f"{run_name}/sp_from_top2000.h5ad")
adata

/home/wqzhao/anaconda3/envs/gene/lib/python3.11/site-packages/anndata/__init__.py:55: FutureWarning: `anndata.read` is deprecated, use `anndata.read_h5ad` instead. `ad.read` will be removed in mid 2024.
  warnings.warn(


AnnData object with n_obs × n_vars = 13620 × 1883
    obs: 'sample', '_indices', '_scvi_batch', '_scvi_labels'
    var: 'mean_expression'
    uns: '_scvi_manager_uuid', '_scvi_uuid', 'mod'
    obsm: 'means_cell_abundance_w_sf', 'q05_cell_abundance_w_sf', 'q95_cell_abundance_w_sf', 'spatial', 'stds_cell_abundance_w_sf'

In [ ]:
adata.obs['sample']

B4_10x14    B4
B4_10x15    B4
B4_10x16    B4
B4_10x17    B4
B4_10x18    B4
            ..
A2_9x26     A2
A2_9x27     A2
A2_9x28     A2
A2_9x29     A2
A2_9x30     A2
Name: sample, Length: 13620, dtype: category
Categories (36, object): ['A1', 'A2', 'A3', 'A4', ..., 'G3', 'H1', 'H2', 'H3']

In [ ]:
from glob import glob

samples_path = [i for i in glob("../../data/stnet/*") if not i.endswith('ipynb')]
samples_path

['../../data/her2st/B4',
 '../../data/her2st/F3',
 '../../data/her2st/A4',
 '../../data/her2st/A1',
 '../../data/her2st/C6',
 '../../data/her2st/E1',
 '../../data/her2st/G1',
 '../../data/her2st/G2',
 '../../data/her2st/H1',
 '../../data/her2st/B3',
 '../../data/her2st/D2',
 '../../data/her2st/F2',
 '../../data/her2st/C2',
 '../../data/her2st/C5',
 '../../data/her2st/B5',
 '../../data/her2st/C1',
 '../../data/her2st/D5',
 '../../data/her2st/B6',
 '../../data/her2st/G3',
 '../../data/her2st/E3',
 '../../data/her2st/H2',
 '../../data/her2st/A3',
 '../../data/her2st/D6',
 '../../data/her2st/A5',
 '../../data/her2st/C3',
 '../../data/her2st/H3',
 '../../data/her2st/F1',
 '../../data/her2st/C4',
 '../../data/her2st/D1',
 '../../data/her2st/D3',
 '../../data/her2st/A6',
 '../../data/her2st/B2',
 '../../data/her2st/B1',
 '../../data/her2st/E2',
 '../../data/her2st/D4',
 '../../data/her2st/A2']

In [ ]:
import scanpy as sc

adata = sc.read('../../data/breast_sc/cell2location_res/stnet/sp_from_top2000.h5ad')
adata

/home/wqzhao/anaconda3/envs/gene/lib/python3.11/site-packages/anndata/__init__.py:55: FutureWarning: `anndata.read` is deprecated, use `anndata.read_h5ad` instead. `ad.read` will be removed in mid 2024.
  warnings.warn(


AnnData object with n_obs × n_vars = 13620 × 1883
    obs: 'sample', '_indices', '_scvi_batch', '_scvi_labels'
    var: 'mean_expression'
    uns: '_scvi_manager_uuid', '_scvi_uuid', 'mod'
    obsm: 'means_cell_abundance_w_sf', 'q05_cell_abundance_w_sf', 'q95_cell_abundance_w_sf', 'spatial', 'stds_cell_abundance_w_sf'

In [ ]:
adata.obs['sample']

B4_10x14    B4
B4_10x15    B4
B4_10x16    B4
B4_10x17    B4
B4_10x18    B4
            ..
A2_9x26     A2
A2_9x27     A2
A2_9x28     A2
A2_9x29     A2
A2_9x30     A2
Name: sample, Length: 13620, dtype: category
Categories (36, object): ['A1', 'A2', 'A3', 'A4', ..., 'G3', 'H1', 'H2', 'H3']

In [ ]:
from tqdm import tqdm
import os
import pandas as pd

columns_order = pd.read_csv("../../data/stnet/23209_C1/cell_ratio.csv", index_col=0).columns

for sample in tqdm(samples_path):
    idx = sample.split('/')[-1]
    sub_adata = adata[adata.obs['sample'] == idx]
    cell_ratio = sub_adata.obsm['q05_cell_abundance_w_sf']
    index_list = list(cell_ratio.index)
    cell_ratio.index = [x.split('_')[-1] for x in index_list]
    cell_ratio.index.rename('spot_id', inplace=True)
    cell_ratio = cell_ratio[columns_order]
    cell_ratio.to_csv(os.path.join(sample, "cell_ratio_from_top2000gene.csv"))

100%|██████████| 36/36 [00:01<00:00, 25.63it/s]
